In [1]:
import sys
sys.path.append("..")
from app.Core.injury_engine import InjuryEngine

ie = InjuryEngine()
from dataclasses import dataclass

@dataclass
class InjuryStatus:
    region: str
    severity: str  # 'green', 'yellow', 'red'


print(ie.classify("I have knee tendonitis"))



InjuryStatus(region='knee', severity='yellow', description='i have knee tendonitis')


In [ ]:
from typing import Dict, List
from app.Core.schemas import InjuryStatus
import re
from dataclasses import dataclass

@dataclass
class InjuryStatus:
    region: str
    severity: str  # 'green', 'yellow', 'red'


class InjuryEngine:

    RED_FLAG_TERMS = [
    "fracture",
    "torn ligament",
    "acl tear",
    "acl rupture",
    "rupture",
    "dislocation",
    "post surgery",
    "post-surgery",
    "cannot walk",
    "cant walk",
    "unable to walk",
    "cannot bear weight",
    "cant bear weight"
]

    YELLOW_FLAG_TERMS = [
        "tendonitis",
        "tendinitis",
        "strain",
        "sprain",
        "chronic pain",
        "flare up",
        "tightness",
        "mild pain"
    ]


    REGION_MAP = {
        "knee": ["knee", "acl", "meniscus"],
        "shoulder": ["shoulder", "rotator cuff"],
        "back": ["lower back", "lumbar", "spine"],
        "ankle": ["ankle", "achilles"],
        "wrist": ["wrist"],
        "neck": ["neck", "cervical"]
    }


class InjuryEngine:

    def classify(self, injury_text: str) -> InjuryStatus:
        text = injury_text.lower()

        # Detect region
        region = "unspecified"
        if any(k in text for k in ["knee", "patella", "meniscus"]):
            region = "knee"
        elif any(k in text for k in ["shoulder", "rotator", "ac joint"]):
            region = "shoulder"
        elif any(k in text for k in ["back", "spine", "disc", "lumbar"]):
            region = "back"

        # Determine severity
        red_flags = [
            "fracture", "rupture", "tear", "torn", "dislocation",
            "severe", "broken", "surgery", "post-op"
        ]
        yellow_flags = [
            "pain", "sprain", "strain", "swelling", "ache", "tenderness"
        ]

        severity = "green"  # default
        if any(word in text for word in red_flags):
            severity = "red"
        elif any(word in text for word in yellow_flags):
            severity = "yellow"

        return InjuryStatus(region=region, severity=severity)


    def classify(self, text: str) -> InjuryStatus:
        text = text.lower()

        region = "unspecified"
        if any(k in text for k in ["knee", "patella", "meniscus"]):
            region = "knee"
        elif any(k in text for k in ["shoulder", "rotator", "ac joint"]):
            region = "shoulder"
        elif any(k in text for k in ["back", "spine", "disc", "lumbar"]):
            region = "back"

        severity = "green"
        if any(word in text for word in ["fracture", "rupture", "tear", "torn", "dislocation", "severe", "broken", "surgery", "post-op"]):
            severity = "red"
        elif any(word in text for word in ["pain", "sprain", "strain", "swelling", "ache", "tenderness"]):
            severity = "yellow"

        status = InjuryStatus(region=region, severity=severity)


                # Region detection
        for region, terms in self.REGION_MAP.items():
            if any(t in text for t in terms):
                status.region = region

        # Severity detection
        # RED takes absolute priority
        if any(t in text for t in self.RED_FLAG_TERMS):
            status.severity = "red"

        elif any(t in text for t in self.YELLOW_FLAG_TERMS):
            status.severity = "yellow"

        elif status.region:
            status.severity = "yellow"


        status.description = text
        return status


In [3]:
engine = InjuryEngine()

text = "I tore my ACL and have severe knee pain"
status = engine.classify(text)

print(status.region)    # Expected: "knee"
print(status.severity)  # Expected: "red"


TypeError: InjuryStatus.__init__() missing 2 required positional arguments: 'region' and 'severity'